# Tiled attention with an online softmax (FlashAttention forward)

FlashAttention never builds the full $T \times S$ score matrix. It walks over key/value **blocks**, keeping for each query row a running max $m$, a running normaliser $\ell$ and an unnormalised output accumulator. When a new block raises the max, the old state is rescaled. Implement that forward pass in PyTorch.

Write `flash_attention(q, k, v, block_q, block_k, causal=False)`.

## Shapes
- `q`: `(B, H, T, d)`, `k`: `(B, H, S, d)`, `v`: `(B, H, S, d_v)`, float64. Scale $s = 1/\sqrt{d}$.
- `block_q`, `block_k`: tile sizes, positive ints that need not divide $T$ or $S$ (the last tile is shorter).
- `causal`: only used with $T = S$; query $i$ may attend to keys $j \le i$.

## Algorithm (per query tile $Q_b$, rows $r$)
Start with $m_r = -\infty$, $\ell_r = 0$, $O_r = 0$. For each key tile $(K_c, V_c)$ in order:
1. $S = s\,Q_b K_c^\top$ (apply the causal mask inside the tile with $-\infty$).
2. $m^{\text{new}}_r = \max(m_r, \max_j S_{rj})$, $\;P = \exp(S - m^{\text{new}})$, $\;\alpha_r = \exp(m_r - m^{\text{new}}_r)$.
3. $\ell_r \leftarrow \alpha_r \ell_r + \sum_j P_{rj}$, $\;O_r \leftarrow \alpha_r O_r + P V_c$, $\;m_r \leftarrow m^{\text{new}}_r$.

At the end, $O_r / \ell_r$ is the attention output and $\mathrm{lse}_r = m_r + \log \ell_r$ is the log-sum-exp of row $r$'s scaled scores (what the backward pass stores). With `causal`, skip key tiles that lie entirely in the future of the query tile.

## Output
A tuple `(O, lse)`: `O` is `(B, H, T, d_v)`, `lse` is `(B, H, T)`. Loop over tiles in Python, but vectorise within a tile (over batch, heads and rows). Never allocate a `(T, S)` score matrix.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def flash_attention(q, k, v, block_q, block_k, causal=False):
    """Return (O (B,H,T,dv), lse (B,H,T)) without materialising the full score matrix."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[1.0, 0.0], [0.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0], [2.0], [3.0]]]], "dtype": "float64"}, 1, 2], "e": {"__tup__": [{"__nd__": [[[[2.0], [2.203336278039358]]]], "dtype": "float64"}, {"__nd__": [[[1.6206211390568515, 1.6206211390568515]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0], [2.0], [3.0]]]], "dtype": "float64"}, 2, 2, true], "e": {"__tup__": [{"__nd__": [[[[1.0], [1.6697615493266569], [2.255234765226831]]]], "dtype": "float64"}, {"__nd__": [[[0.7071067811865475, 1.1079403076572498, 2.1004053012281645]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.0012301533574825742, 0.2987455375084699, -0.2741378553622176, -0.8905918387572742], [-0.45467078517172255, -0.9916465549964624, 0.060143602597438485, 1.3402152455545335], [-0.49220651855132963, -0.6204748998199404, 0.4898420501851982, 0.35688700816006075], [0.10541424899789856, -0.9304680447082047, -0.02925182246327349, 0.6953031944582878], [-1.344214547285082, -0.45761576104021817, -1.901222739800844, -1.289537739784976], [-1.8417350377917323, -0.23509113107468127, -1.2674464814437032, 0.2712643588217015], [0.15675108662422516, -0.18693094462995438, -2.516759710820513, -0.5386928958466366], [-0.048500945401071985, 0.11330898600330756, -1.5301357655053935, -0.47775327603393064]], [[-0.9785190780566395, -0.8088372394255993, 1.0608986233860787, -0.8075346753318965], [-0.0325217049455206, 0.8843898673831739, -0.583600432743302, -0.11170194958415963], [0.11046414324948059, 0.06378177425506196, -1.2250558264176934, 0.0761402303770081], [1.3588234217415376, -1.5471446781284823, 0.8593826880215982, 0.11935402569658124], [-0.6414703941072214, 2.000416546342423, 0.7622597120847118, -1.1992889021052233], [0.07451622877146342, 0.5766895836701853, -0.1887821253507493, 0.682910267195206], [-0.06651732014941557, 0.6672475608343279, 1.438522591656152, -0.6756622510056528], [0.20313861038960904, -0.46330757653841514, 0.12726841122583082, -1.18719452785014]]], [[[-0.5793015965026732, -0.1961959728044967, 0.8987638721004078, 1.145222007454132], [-1.323527792484255, -0.7946423659870495, 0.6469034225734218, -1.9924197841744944], [-0.46316986495236695, -0.09728692567008902, 1.2570149772868198, 0.6894039005707556], [-0.32721342022219785, -0.3685758940999591, -0.25019540051792494, 1.5235294004561601], [-0.4280249425728672, -0.3036803883647294, 0.35258906728526535, -0.12077044508645512], [-0.19728422796572256, -1.1140671431510563, -0.011521468038548173, -0.4435812229744192], [1.1661277761902227, 0.6530885027011638, -0.024143613009932233, 0.6683810232673438], [-0.3398695517131494, 1.052126358426947, -0.005399560671626605, 0.5833823541804138]], [[-1.2908932453234871, 0.34668004887842974, -1.6882041173665416, -2.0353289449399323], [-0.3044768777114372, -0.8999276075985952, 0.16405279571222256, 2.2447566264860495], [-0.8317231814120817, -0.6239435864439059, 0.2054039460646989, 0.49301329141235634], [-0.1764060659057582, -0.20593033025321647, 0.7024629551205442, 0.5199076370338984], [-1.0336758320736887, -0.07918131861584184, 0.035286848661474135, -1.0544846220491104], [0.25983910067436333, -0.8579564771765439, 0.9720667079170427, 0.1927459126050724], [0.08930648576905029, -0.591028352856274, -0.11860982387769403, -1.9977462929070549], [-1.1314074705230586, 0.3628397991887543, -2.1285670418221447, 0.8466085214811634]]]], "dtype": "float64"}, {"__nd__": [[[[-1.7460964753739088, 0.7567385026642676, -0.8454970328793241, 0.7789910843424612], [0.1309512075847998, -1.5368349402914887, 1.2491487495584548, 1.4417071555226115], [-0.0658049060002071, -0.27391627217232034, -0.15986696597063635, -0.9751523227787462], [1.0985867597569177, -0.5428919317301868, -0.051190412691676575, -0.7932964032030436], [-0.6260730997201972, -1.2777251516511705, 1.2570693137143927, -0.15408757320601318], [0.9659216187288089, 0.01332459691325976, -0.6944035277028942, -0.3266852600022522], [-0.5602310505028996, 0.007959099184913658, -0.3752668396769557, -0.29992171594179834], [-1.3785746841359137, -0.8068459276211205, 1.65405754880043, -0.6712332162517173]], [[-1.0540937876234928, 0.33732633257503086, 1.407272199556532, -1.454024302481208], [-0.20852184825127065, -0.6320525539349078, -1.7610194743274028, 0.7349267092583915], [-0.02344391855484465, 0.07144184429260535, -0.7523114724455892, 0.4547841629969166], [-0.539297349487321, -0.1429032084967607, -1.108260792181513, -1.2161027602081953], [1.3355318553000226, -0.5071047045384752, 0.2916803563558019, -0.03379043476737232], [-0.44114520276218416, -0.5079609703372195, 0.6300825914455861, -0.3018676045339098], [-0.15144364817129202, 0.022221557194479512, 1.1765083202520015, 0.6805109746331383], [0.3826002677279928, -0.5635713933532417, -1.3819687200064654, 0.949529931735302]]], [[[0.9664471342208761, -0.1407083991233403, 0.541883841603518, 0.781443053885405], [0.8311844526649818, 0.9213834732464707, -0.45561765066113274, 1.5149729826647176], [-1.2465930582857083, 0.861723076993649, 0.4939320790307585, 0.8736191135245024], [1.8790083070967372, 1.484445473378649, -1.145176907189325, -1.688671580267857], [0.8168890590537966, -1.015012415875923, -0.0124053886475133, 0.839727477005194], [-1.6437973511855861, -2.1099804543094787, 0.259299020337227, 0.04438588961329704], [-0.24580307138192253, 0.03853476377021816, -0.8605156073672797, -1.5134944072721068], [-0.1666548508803217, -0.9717086910493917, -1.643481322411107, 0.5056810713932767]], [[-0.061398628422175805, 0.40652853663281385, -0.9892949414259369, -0.6580587918366578], [-0.9990430272201656, -0.8866418670580481, 0.19540791774940214, -0.7829746163349458], [0.3560662888202746, 0.3397559254182162, 2.0251609868801026, -1.3927890458668095], [0.8879023778350454, -0.0894879618884183, -0.01402973013099694, -1.4498638219066111], [-0.4601938127246543, 0.743197263443954, -0.08247837201730518, 0.08105437041126562], [-0.29071668060054134, 1.1545697469862868, -0.021472567043494796, -2.200415672482062], [-0.6920725504744906, -1.968796607080243, -3.2514384154965383, -0.5301153497723121], [1.3335598501027237, 0.04711990613059292, -1.1725457074049794, -0.9406998682024224]]]], "dtype": "float64"}, {"__nd__": [[[[1.1306132302500087, 0.15762662339846478, 0.04799924156205696, -0.05346178883805718], [0.038400261555346496, 0.8054056469437983, 0.5525672973560755, 0.21570470002449457], [-1.0428683575900106, 0.511108764909727, -0.6842470779924941, 1.0938456759004787], [-1.2710508217241274, -0.13762097627558853, -0.007358286291270949, -1.3246455506441366], [1.721971643856479, 1.4604067672595522, -0.46358376160908316, 0.7717211655645232], [0.37867606967021183, -2.613559463345258, 0.2503980162775905, -0.061344079833248494], [0.08321735347453037, -1.0768749198271783, -0.2693470462204049, -0.17825876338229713], [1.1880942172123188, 0.33442706039101433, -0.005555030202663769, 1.5289699979109401]], [[-0.5552479297636472, -0.3894303048753847, -1.8167550113390598, 1.5691058462836123], [0.9643323681944285, 0.9168480445117076, 0.6688984464469374, 0.110148587663652], [0.21548893918232892, -0.2520065902673078, -0.20360042572294623, 0.05430361171369427], [1.5118295115637772, 0.5556879018080786, -0.058460125758861455, -0.5793920504433961], [-0.6349984874116595, 1.6027050111724899, 0.5066873042271038, 0.06755055831065467], [-0.34618184215272274, -1.1090534117005277, -0.06686155746989603, 0.873658034470201], [-0.39253724683749003, -0.2272425830005601, -0.22103435143897837, 0.10959436799816868], [-1.593010878745609, -0.235398351965091, -0.8543955181540714, 0.8845850325626053]]], [[[-0.7705986765648598, 0.5770467473006955, 1.5244374527267601, -0.3135962612816255], [-0.6015762521234661, 0.1914322591260928, -0.0020290276090286124, -0.9936163473397114], [0.4609194157133795, 2.015516053475408, -0.25811199526572426, -0.2028761960646632], [-1.0449320235690858, 0.3190884516865476, -1.2469762247076281, -1.1069310243514836], [1.2796672718437896, -0.9054530593946442, 1.0813575693313382, 1.524358003782807], [0.2593264574767428, 0.5533914028508236, 1.9522509436867783, -0.19672840206571665], [-0.5930057969416156, -1.3532310647829418, 0.041707080346346355, 1.4791442109183066], [0.9595953814008653, -0.9420912414179574, -0.855375383832596, -0.5041712293409636]], [[0.2922680824953417, -0.20531142710368377, 0.21445394544308127, 0.2967393780454473], [-0.2987735866784475, -0.040173941602943734, 0.20659237721603418, -0.08396970353317602], [0.5035208649940508, 1.8708757733280048, 0.5919722847708305, 0.05581054640855185], [-1.6861186233619148, 0.38795708601520174, -1.9466784117672074, -1.409034086156122], [0.8546392524687652, 0.7062350601366643, -0.14993899834888477, -1.7100111061522156], [-0.37134851689085024, -0.678738318216024, 0.6368407461302101, 2.2577305324275314], [0.21693048795465586, -0.7793111280321169, -1.1705515303032201, -0.05609428076827262], [-0.17679319351358783, -1.1515191595274716, 0.1163552356921842, -1.1509135213108617]]]], "dtype": "float64"}, 4, 4], "e": {"__tup__": [{"__nd__": [[[[0.09117370348786395, -0.26754756381282463, -0.14104644553116405, 0.2023545419902443], [0.45716531171274577, 0.4343857094055813, 0.13033078573090728, 0.33044006115792224], [0.5563512535176343, 0.35359689179731624, -0.01998032237819997, 0.4815131661498273], [0.324160911095679, 0.2294645237301529, 0.053330681244966115, 0.268832678419944], [0.2719925178906475, -0.22398497943803264, -0.165915703907826, 0.2158724201921894], [0.7566155856518495, 0.019211122974015254, -0.05275797547673756, 0.1657607952277041], [0.031874015403831946, -0.7667847232302798, -0.07834830977502706, -0.036540583980880954], [0.09635284984028053, -0.5372674461615295, -0.10177580146880028, 0.040873999356756806]], [[-0.24374905922791523, -0.24249795492104195, -0.8125579799722404, 0.835035870862386], [0.04733286344810255, 0.15543728429215348, -0.23245217612179217, 0.30526302731005545], [0.07761468584463399, 0.2823363690596897, -0.0842961568885629, 0.24809546902584328], [-0.41454163510245323, 0.4587509788947665, -0.01703070459193999, 0.33228506340331565], [-0.2599102782594058, -0.2365914008102562, -1.144611207279596, 1.0033762739941432], [-0.15214770467450386, 0.12875670742721798, -0.19100295361996047, 0.3442559745137513], [-0.3456089032104011, -0.1923191084193844, -0.867875766314974, 0.8617842603260861], [-0.015632415738845787, 0.10418635393388981, -0.40691215981529555, 0.45609066216018784]]], [[[0.19353921956563439, 0.5289992658989308, 0.6527573934379077, -0.07364451941739505], [0.0006346212105245726, -0.02280061110298187, 0.9782415565838796, 0.2707022501805146], [0.15446299218580858, 0.5552879620598863, 0.6830273925643039, -0.048645819487644404], [0.242075358756396, 0.15254112899373706, 0.4455123742146584, -0.11269676018922135], [0.12169811342838134, 0.16255012102704391, 0.5899260012728401, 0.04706966577803719], [0.21239764814725615, -0.06227550725180211, 0.7639273415531687, 0.11968575232671518], [-0.31470817908374094, 0.14425418343837035, 0.0761852399077405, -0.3912621895890259], [-0.07779245868848424, 0.4804399729931157, 0.06801919855172074, -0.2748273454936069]], [[-0.022038885197077755, -0.5732774502610638, -0.47644170609772274, 0.36895510785818453], [0.15591810088986185, -0.0754686551939503, -0.3703095738907525, -0.4935114978177175], [0.018921597634594237, -0.005619889431833867, -0.2442249692778807, -0.2559481704309401], [0.012825663315848486, 0.35572023913988543, -0.051841741971639675, -0.3021311981897209], [-0.12357767383435542, -0.03784959209502193, -0.05758941825224957, 0.2269328652633586], [-0.07153131274190136, 0.47153247740422993, -0.07776383385447361, -0.29094414239757876], [-0.3003516230876805, -0.09512187135459578, -0.22856970057308998, 0.14015622884071743], [0.19934704777291662, -0.6384208721150133, -0.9129776821456183, -0.12325500788114167]]]], "dtype": "float64"}, {"__nd__": [[[2.1107043618785086, 2.602603696037291, 2.472478665846052, 2.3541357824350246, 2.53603447728615, 2.74062024440946, 2.3091960929163613, 2.1424490495207476], [2.547743423107409, 2.068425180750917, 2.415462559912097, 2.3731813366278858, 2.5672320096343464, 2.0633088986105315, 2.4121415833885362, 2.24079945291388]], [[2.3492971812086796, 2.627094101114, 2.188309441441265, 2.5311412322713105, 2.0776401965509512, 2.3443590671506165, 2.5285224046017403, 2.2148243550203266], [4.084659585276228, 1.324569221954491, 1.9348895339450554, 1.7701776306527788, 2.683368005712611, 1.9558605733236578, 3.2512410817342188, 3.5076112602816436]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[1.1121072717293008, 1.0626506057881553, 1.0847512652964209, -0.4740504706085282, 0.5145205633180888, -0.13206950521935684, -0.38881206140248165, -0.3391456571420661], [-1.2997152730341506, -1.4438636958951212, 0.7943147908760457, -0.19123540391952007, 0.21642485112762316, 1.0017103884541199, -1.7331324950160742, -0.784130073815894], [0.17533484346507067, 0.39209478666557296, -0.3770744504010968, 1.0291792326978486, 0.2103949388296369, -1.2133821388278598, -0.9307637759906452, 0.8054710604946225], [0.4638314564560598, -1.899061239006601, 1.3477123038624088, 0.5980270310938874, 1.3433429876964984, -0.383700881075084, -0.2957062213323259, -1.126457084751319], [2.5369295701318877, -0.17545422540714875, 1.5875331920819455, -0.6472924513872795, 0.16384119233310074, -1.67135248657008, -0.382867148766769, 0.9837549084000277], [-1.2517438155744967, 1.072227544934612, 0.3372378158484627, -1.0439509249842558, -0.5015972714256735, -0.4590656767054133, -0.04951933794250218, -0.5361439659430923], [-0.8272879008045365, -0.30458770251481737, -1.026889738609049, -1.289526572643077, -0.04817648250735862, 0.8828742255545683, -1.529371705117365, 0.0035081514639604466]], [[-0.6499561948012956, -0.9771454371346897, 0.8534376527146189, -0.5181696890906662, 1.4983016830648794, -0.7798393641463933, 0.38650199362583243, -0.22728300655720315], [-0.7540218091263632, 0.5876750425777528, -0.15498257307480726, 0.6032105807136467, -0.047291507216996057, -1.0858162553014383, -0.10206924318890695, 0.05195516951614051], [0.9584629538811972, -0.9062690073351785, -0.03932985326730542, -1.721878730230039, 0.6514930603433157, -1.0814829746406458, -1.8063595210836054, -0.0592168342747372], [1.1056849586628463, -1.5244469482118754, -1.0880355853884272, -0.7432748910532813, -1.1294966463420033, 0.37942788833458474, -0.8073674450444858, -0.7215147364602021], [0.5833053986823111, -0.7555146309686842, 0.43278009778072224, -0.9713851718820432, -1.2121385142152894, -1.8354492891909227, 1.861202786911729, -0.32025903213099666], [0.24393336794842915, -0.0310495089807992, 0.15996175700726273, 0.04968647937437248, 1.9082170392026987, -1.0389265909571472, -1.557462106098177, -1.0119596684843548], [-1.3347090097585461, 0.7469620524819425, 0.8203782095377569, -0.9613155451503265, -1.3904357764629918, -0.35479665314109476, 1.3911238778763935, -2.819569417798316]], [[0.5266267690158747, -1.0757552274485853, 1.0403671465412743, -1.077923940876752, -0.2854158272158405, -1.5062791006395633, -0.9772578504980973, 1.3858391630461182], [0.8205786858900587, -0.40166815215067564, -0.8701723702042559, -1.893807136511814, -0.3936567119009843, -0.030903252586298, -0.08405435783448403, -0.0937919279761702], [-1.121809627226926, -0.06627388536561797, -0.038673379535105926, 1.2905620378031775, 1.8667334543523622, -0.13698902691779163, -0.7663059511758866, -0.06498212976196366], [-0.607621275254437, -0.7424338157462491, -0.05865015192214237, -1.0433044311523152, 0.6061067511550176, -0.10391554918407729, 0.25000830685995257, -0.18294003597067152], [-0.7272544472524123, -0.9479595597488518, -0.23727600432454096, -0.5487603374278119, 0.23390133320957562, -0.004431703197639366, -1.3623067479227293, 0.06712129942835174], [-1.3428604171407572, -0.6164776255192805, -0.2943536092178474, -2.075273277911311, 0.09150744785998134, 0.15098334971433505, -0.15802419342467286, -0.4243172724715942], [-0.37360366010814, -0.9765394686110568, -0.2697003697288119, -0.5526118035148618, 0.09168539268739877, -1.2041360895448572, 0.235589909011144, 0.1432188934776698]]]], "dtype": "float64"}, {"__nd__": [[[[-0.1415628321787237, -0.439204823086289, 0.5523398924701125, -1.6646017067938645, 0.4604533381451202, 0.24303746619037217, 0.28337629253461555, 0.3833125924326753], [-0.6536444065928503, -0.25946005830143554, 0.6370583312163494, 0.4306392440831103, 0.20671276544264874, -1.5143202451196276, 0.5378871148832081, 1.1694709919931063], [1.0096714930428576, 0.2338780487691054, -1.5576786036041945, 0.9425448142280201, -0.1472546367197673, -2.5325196480669656, 0.37720740310720396, -1.4921717830795365], [-1.296440580195063, -0.6349588923738527, 1.272591177882884, -0.37084535417839304, 0.2709915877367156, 1.7479679637522305, 1.5940297626557918, -0.10335341582736501], [-0.2415210220605113, -1.2608706340895501, -0.6944580741815911, 0.4253583476127701, 0.39573076180964684, 0.11023823117395956, 0.9948016575639254, -0.7723680130331807], [-0.05607652018346502, 0.7312429884948122, 0.584145906227918, 1.0709457757324194, 0.3970204067650606, -0.3094025555968319, 0.3621922602623995, -1.0025951963841089], [-1.6394593315431263, 0.5806717808971771, -0.0551403328210084, 0.30840962641700853, -1.697684363841418, -0.36511402472543003, -0.5998571137315275, -0.8642121147366698], [-2.25501500646246, -0.33487348213705126, 0.8973042881661908, 0.3809977218637382, -0.6009466096658063, -0.014875971650300183, 0.7568000802380088, -2.7604178626541933], [-0.12453654680663752, 0.5432001471814378, 0.6821286252341751, 1.7007302070100299, 1.1350456790964805, 0.3125628347462472, 0.30197600611542663, 0.7869453479945998], [-0.5393269947423406, -0.040188054943185156, 0.9062390030785368, 1.9574452148950907, -0.159267305731742, -0.04840068932986404, 0.19848019378767898, 1.3432036940080567], [-0.03031330358915025, 1.469359036471145, -0.9666572174287505, -0.18603629573699512, -0.19816515668552595, 0.78650572409622, 1.0452603769782782, -1.5094443581182324]], [[-0.9151176258112247, 0.3368524787764621, -0.6586936458858148, -1.522427775647643, 1.0384799720234792, 0.49397008415364824, 0.4931774968607472, -0.4754457164460954], [1.028938452167321, -0.23995999642712024, 1.0964352778229522, -0.9116651509426222, -0.8541354476838009, 0.20466321059769635, -0.7023321457970587, 0.6754510653688304], [0.26008774412905367, -0.9229712178630985, 0.07286560094510024, -0.35127768818736665, 0.9158328291770471, -0.6325529831782043, -0.43917309547287997, 1.2112369053763345], [2.238593389242513, 1.999008664227936, 0.06322301166060816, 0.21885470706822427, 1.5334586622826605, -0.12443480624463125, -0.9763491371373664, 0.11687559743629844], [0.45151678053213146, -0.8291496178182481, -1.6462311414991517, -1.4367295145541013, 0.665429284172329, -0.7583765163118437, -0.14134709237111392, 0.21255868433783848], [0.6191408790947098, -0.3349027987134741, 0.49871570048060176, -0.8900602921208429, -0.36171402567801186, -1.0244155877878707, 1.1317672163619168, -0.027083228066991226], [-0.7393074876302995, -0.3523717980651465, -0.22172848072327495, 0.7005197913250095, -1.595636178185236, -1.037205841625419, -0.3781072515574971, 2.5323058327678063], [0.9556788925319947, -0.11145167913094357, 0.7119995824108395, 2.057436169630336, -0.23374392677539008, -0.36641334409756016, 1.2119435432218957, 0.49417549813294265], [0.6713331553225568, -0.5082223992805806, 1.9242031323427422, 1.7095908015936603, 0.5659477888727499, 0.6842898381472375, -2.0272608722430356, 0.6377097934972082], [-0.19410858722532306, 0.43390570582822213, 0.6824594118460978, -0.3412917681082754, -1.6904818001870843, 0.36787456507423494, -0.7413836524403951, -0.3301940927243394], [-0.6045098440941945, -0.34157425525518803, -2.3095209572659416, 1.2169471272315173, 0.253346102326474, 1.111399262918864, 1.9798418298099358, 0.022593122562750742]], [[-1.8024362558405755, -0.8938868503795847, -1.2069044808967102, -0.501874643144999, 0.07964797806597748, -2.002187267023239, 0.34247549334155225, -1.5103566532858779], [0.29736526280001135, -0.1094911451769814, -0.3136312391925775, -0.07304305594131319, -0.5397780157316924, -0.6124714434041737, -1.6827563908688876, -0.02962731484932925], [1.8455337106233174, 1.9804950628846045, 1.3218175965449428, 0.7058016008515581, -0.6764801240799883, 1.4427507882162613, -0.056127554440968945, -0.06887550976307337], [-0.29122713183926474, 0.09196189901829563, -0.43521490158805637, -0.08378216937310629, -1.0845991087272615, -0.3736637047405186, 2.280556469949822, -0.0695940279619468], [-0.23866056477176292, 0.5323742791050526, 0.708121821703462, -1.1137688591720734, -0.20715878280324507, 0.9182758409347652, 0.26983762907210557, 0.12195404350177193], [1.5540176686087332, -0.6777727596396438, 0.08274589510763317, -0.5200586652550714, 1.4903082290762408, -1.9518603992765322, -0.6701416954837595, -0.5291569660279176], [0.66353962317159, 0.606275476109527, 1.3951942602387315, -1.5740328423373253, 0.7504750090304797, -0.29327986428830616, -0.6656252372581756, 0.539568533372521], [-0.9201395827356486, -2.0769132953143226, -0.37021848892263526, -1.4977708516419035, -0.6489158210706784, 0.3715547637511007, 0.311999745614158, 1.5868994489785668], [-0.19965474694348667, -1.5333315905478553, -0.7560968566356783, -0.9193842246262718, -1.2178251584582491, 0.43536719501544086, -0.6452323379652398, -1.9773728166870077], [0.6969228031192131, -0.11142656870670702, 0.3568372783525386, 0.10572154469998292, 0.6316547341483746, 0.038018747015535724, 1.2362024834782115, 0.42498278997722555], [0.3920676233884034, 0.40986691768255656, -1.45663774256872, -0.16463575180371992, -0.2593164895494978, 0.2075467211604842, -1.3586941642962918, 1.634114776892911]]]], "dtype": "float64"}, {"__nd__": [[[[0.10438866415652721, -1.211317540650674, -1.7088869635788377, -0.28128729372066874, -0.08967541292646578], [-0.7182920859743898, 0.09211382792392302, -0.6411757672395606, 0.5516582517062987, -0.7245418183888837], [-0.03850022510492524, 0.9788391733621384, 2.5716688829102927, -1.0076431926306066, -0.46450777008709926], [-0.8398356977999346, 0.7843283217692749, -1.1480993057175632, -0.4843570455930865, -0.029601283237510274], [-0.9786785394797441, -0.9573250423091814, -0.47562288812186493, -2.10044024033736, -1.4454782170733342], [-0.4130340877931743, 0.14823063634199898, -0.18575998595359344, -1.7739686948393945, -0.46378629175999775], [0.7984376405061764, 0.5558655119913222, -0.07874786152783442, -0.8873698750982034, 0.6312163709522058], [-0.5789289812099466, -1.1692484024740184, -0.8021862351617889, 1.4483452917741528, 0.22018003820169318], [1.1592470530477048, -0.47933631330936793, 0.938149339185143, -0.6015040436180767, -0.15742736903283047], [2.4863502599556933, 0.7671486322996173, -0.5011703753276925, -0.08481903902671983, 0.32622342088564454], [1.2104575895159044, -0.48913309167015023, -1.7413190846044793, -0.2796418131263074, 0.015385457163756576]], [[0.10953796873523998, 1.3166901361826624, 0.3166870783059282, 0.8129169716359846, -1.1011136939248025], [0.8679014884245477, 2.096219831656284, 0.7729092902501778, 0.25355571994071757, 0.15398818840774314], [1.7871422390260032, -0.9271814833908629, -0.11110235795404999, 0.4602075929382135, 0.744001107813199], [-0.43732790570838587, 0.3073213947272354, -0.2779214813511223, 0.12047113280127958, -0.13209154037389578], [-1.1415936778555495, -0.021113585023446714, 0.8771515220618357, -0.9670182915416742, -0.24109174877483452], [0.6647800644571485, -1.0698651067471732, 0.18263833750815855, -1.0601298189534365, 1.1346305008337747], [2.3128213818798624, 2.0222552842457637, -0.21918067777313757, 0.7401991287406264, 0.120997728500156], [0.10210589606928461, 1.5477570609378302, -1.319179576184154, 1.0552945974661148, -0.048975995155734274], [1.4085406206041422, 0.18723336889944325, -0.6726719945261853, 0.27714037675138314, 0.7359670950296834], [0.0357636741111941, 0.4880382765565979, -0.521675175701983, -2.133883900845073, 0.900023583773342], [0.6991597361842888, 0.1481783846143336, 0.06841056221032339, 1.0362957693547372, -0.4571067254765605]], [[-0.7065340653585184, -0.18854986949598598, 1.1890971885515316, -1.387112904264485, 1.1918299926140086], [-0.639252854205514, -1.100743499724022, 1.2600618361423188, -0.09689009479619475, -1.3002343643849232], [-0.3587330590564334, 0.9310524308870369, 1.192074192621773, -0.42709868192105066, 0.406320759097275], [0.7140846323944298, -0.6446264937163554, 0.35503607516644337, -0.03187791475556112, -0.5360244104730933], [-0.49214793537892493, 0.06703448186721092, 0.029853486958632845, -0.5654899533243374, -0.4259451084001599], [1.1133372383403377, 0.21382713517698357, 0.8850943665882824, 1.2018222319374203, 0.5888422596395486], [2.2708863133824324, -0.8253040317638951, 0.8084095930822921, -0.3181439730310917, 1.855783263371278], [1.7004401213924243, -1.9542111820067407, -0.9688662425118804, 0.6643129107943418, 0.7898672098844762], [0.7365642161592768, -0.0708653281733429, 0.45512082456347686, 0.6530214357863233, -0.07787126953732391], [1.0273019683616285, -2.259497249109046, 0.6337935195699923, -1.034052612285726, 0.9586778079693892], [-0.2286844248912001, -0.8887872710840246, 0.3739021567442434, -0.9113334978457667, -0.9127678975553941]]]], "dtype": "float64"}, 3, 4], "e": {"__tup__": [{"__nd__": [[[[0.2604422785748288, -0.12366791552158783, -0.3289034742107956, -0.5836470053308922, -0.19899347107407178], [-0.1198667162981355, -0.33469669156348597, -0.6855727939266993, -0.013840617604451827, -0.005343230524129854], [0.4651023016760906, 0.25103679801854634, 0.35288130499122133, -0.5420429531837457, -0.23987345302304597], [-0.11262141829884394, -0.2586986916456934, -0.30504460463196303, -0.4614184877975456, -0.3201190116685592], [0.14374706514883592, 0.03218274423220786, 0.009849965755048483, -0.41499105013988685, -0.3459216580733693], [0.12370310659213941, -0.21729267713408007, -0.5715965664599937, -0.15142358113958124, 0.04718047227127114], [0.17453967460854683, -0.25943821709851134, -0.7243959577975362, -0.47978754557598563, -0.06664465849785098]], [[0.5855738528007017, 0.24475918934005825, 0.037131458985663945, 0.0659466879177715, 0.16524459052480478], [0.7666408511494964, 0.697599546759404, -0.17304151870331996, 0.1582014461464886, 0.1620876487603443], [0.27116323231178113, 0.284314313882801, 0.24606520486799313, -0.20890392931684343, 0.16748677300838602], [0.2846241572307218, 0.48935770295817693, 0.20779444243393821, -0.3133513137216101, 0.168873972539614], [0.5776804196517819, 0.0594939306870384, 0.04525354596705296, -0.3923263585616618, 0.5236621212519685], [0.32285309577128624, 0.26136436449529765, -0.13251750722871058, 0.06566495307138269, 0.14505132860212602], [0.2990831512293227, 0.41577273657393915, -0.0408330738500133, -0.5840171129143382, 0.2526964475321535]], [[0.9885152098271174, -0.8075990726548699, 0.4385932601247621, 0.14259362302807307, 0.5152668163218244], [0.6674601601886848, -0.682157785381087, 0.39827361438780967, 0.020454917375365037, 0.24928737729622394], [0.32968116585287127, -0.5771496791778058, 0.7431274156496839, -0.29972210732838267, 0.3621155932384413], [0.614502492613008, -0.7335765272689615, 0.4136071662678764, -0.13638419450987124, 0.4739186507352658], [0.506363203671842, -0.7402969708404211, 0.4458384671073714, -0.058128170655750586, 0.20831456107833804], [0.5796182440946915, -0.7368875303250587, 0.3032800976059415, -0.11153777924153255, 0.4690816327192975], [0.4755666150158159, -0.672963444745158, 0.533184636069085, -0.1884545057400792, 0.4934122834707098]]]], "dtype": "float64"}, {"__nd__": [[[2.2778512604849723, 2.7973039391280636, 2.642806140000631, 2.790777699753491, 2.316290772677378, 2.8716955725228646, 2.143636041018272], [2.5088748734688284, 2.404202560693461, 3.101414650054016, 2.6193306826062126, 2.989455426709788, 2.844408365259265, 2.4562352570480166], [3.1744505766850857, 2.976442658792754, 2.18443040935464, 2.746386999748701, 2.7937580639069015, 3.1723480125336505, 2.8991811751311642]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-1.567294355953092, -0.026707754798113225, 0.4968235216908353, 1.0230259861312425], [-0.14172786178901123, 1.0478552084476458, 0.017960975134801903, -0.09328848097955465], [0.5735629478577897, 1.0583927761257572, -0.34143445445859266, -0.2437616219559586], [-0.16083140469983928, 0.08276912699183876, -0.9004082096051503, 1.0280060600662873], [-0.4004096317783333, 0.4624434567778641, -0.8254725722239247, 0.3588039528088937], [0.3915930157503348, -0.4206814067973341, 2.0208894329971416, 0.3710399612419042], [1.7769289229773846, 0.9591389238981252, -0.6622748070560666, -0.3821868156578954], [0.43610206735723056, 0.06117253183875087, 0.04946114175893247, -0.28620495437455035], [-1.808476355740995, -0.22543264342069147, -2.2181214639227598, 0.3718195687849371]]], [[[-0.726057787427706, -0.7154167261721429, -0.21930379562997881, 0.27267234629586407], [-1.4320061777359552, -1.7486011895534066, -1.0660658714998135, -2.0417275241987234], [-0.9668496098892981, 1.590883030522966, -1.0565859164970517, 0.6514051749001837], [-1.3716348082651353, 0.2991201525724937, -0.31973381221268193, -0.05981278553427285], [0.5687775943675412, 1.7562401253161792, 0.19470634983327353, 0.1243589513960949], [-0.9733677174823148, 0.5833396553323329, -0.24607233054527072, 0.8320070429084299], [-0.0437064504849727, 1.740855703809851, -1.982916503137466, -0.2965993601550742], [0.8814816388012593, -0.35069219708498706, -0.7921730690054608, -0.26588060573958244], [-1.3799287476073374, 0.11895377168976948, 2.440461840173224, 1.145031498226728]]]], "dtype": "float64"}, {"__nd__": [[[[-1.1090091065152694, -0.8733410823282927, -0.40472780902245364, 1.0044161277339365], [-0.8214870091691876, -0.6902314189475338, 0.8847494335149046, 0.8646525704751467], [-0.3738146603894485, -1.1177634143727924, -1.549744823262707, -0.6989901269101042], [-2.230528902271529, 0.7498198819099152, -0.6300306998045586, 0.48129366424513825], [1.8683248876013339, 1.172995707132864, -1.1511348379469908, 0.8692489864766937], [1.1578570275164373, -0.7463565180960922, -0.9532969230720043, -0.1096918973385416], [-1.6014233495161816, 1.470734949301553, -2.4053637930304905, -1.1068074157420604], [-0.2695651640091679, -0.22708709133068325, 0.16612417314262437, 0.2714485686748367], [-0.21361201462544818, 1.1368863187699414, -2.1393761696371945, -0.0001645139169606655]]], [[[-0.7145844117153977, 0.13251343514348632, 0.22075983794627882, -0.911828876494789], [-0.6409489965354924, 0.7925867232518442, 0.34905626865098566, -0.6802484366024778], [2.0398909593419425, 2.309177648495082, -1.4624624789603549, 0.30179496030034325], [2.5089944356323914, 0.7838986291848924, 0.22106014291254644, -0.20805683335943306], [-0.5411942858767205, -0.2125147026197359, -0.5507294124081377, 0.7449083585580341], [-0.3981536526914417, -0.4411515120100576, -1.2021606868290806, -0.04958987438030339], [-0.8941216734390032, -0.1807501687290939, 1.0418155527215323, 0.3659248863111264], [0.5047252947854309, 0.35609089975780356, 0.0591757301914933, -0.12732823067162988], [-0.30780621590166646, 0.759117008833072, -1.0842398092882488, 1.3407225862255443]]]], "dtype": "float64"}, {"__nd__": [[[[0.03407806268481745, -0.7487003905773738, -0.48914607904001295, -0.6769904011881117], [0.16020649902804115, -0.7178430444883082, 1.1421311987837366, -0.7816389926142432], [-2.272521930414174, -0.730998492725895, -2.0085211732996573, -0.0398688505726567], [1.0591579783713116, 0.6477598002656734, -1.3375895049472426, -0.7622552815679994], [1.7786335355666851, 0.3185114087343339, 0.004118692634058828, 1.0560625611935577], [2.4527411254301583, 1.2965732775797387, 0.139415254770579, 0.3546881759848233], [0.6170797910982172, -0.6126984965101239, -1.0587194727174403, 0.05244908348938066], [-0.9472112294631205, -0.06146241903506856, 0.0961230399920905, 2.341037321882936], [-0.8510499946565668, -0.1217889319709347, -0.1643028895845328, 0.4373105333753307]]], [[[1.0467243315836225, -0.4904835016266552, -0.8213520301050313, -1.6407106159316551], [-0.9262044968028467, 0.5364408345765945, 0.045647041823202894, -1.0154339949061588], [-0.3727674764848574, 0.0314729477902652, 0.5036789316089304, -0.5933943528385258], [-0.2504400576578742, -1.818434733340209, -1.1513858022379435, 1.6177156581788448], [-2.1881406726948454, -0.33780255263402675, 0.21919822934425529, -0.3743492453270371], [-0.8081575233653976, -0.061078532387332324, -0.004965401502279523, -0.08211740493189977], [-1.8777121211521157, -0.14740362795539622, -0.8551126902915559, -0.5477358748135395], [0.22697476334145322, 0.6407287504805665, 0.8967773342418733, -0.49641305697888843], [0.9240027767201172, 1.1737275326239913, 1.1366379245460594, 1.3897679388114335]]]], "dtype": "float64"}, 4, 3, true], "e": {"__tup__": [{"__nd__": [[[[0.03407806268481745, -0.7487003905773738, -0.48914607904001295, -0.6769904011881117], [0.10009343881305503, -0.7325497161739778, 0.36466124773866393, -0.7317632686284642], [-0.9719081765522455, -0.7323471324823432, -0.7318369360291501, -0.41805660167449193], [-0.09974917233717188, -0.3016753310968925, -0.8038547154847977, -0.5892608255958824], [0.3532421528177822, -0.055238898376003695, -0.7624480800181502, -0.23600688439915748], [0.4725761171923532, -0.3365495827487654, 0.4054165781207336, -0.4616575874929522], [1.5508030757699887, 0.3297559550517131, -0.16219661138667518, 0.741350751425795], [0.5101591112423435, 0.020791177542952528, -0.3804221660148949, 0.34805774794615785], [0.20692556419747574, -0.3080322526946009, -0.9732340534378805, -0.06385980305610607]]], [[[1.0467243315836225, -0.4904835016266552, -0.8213520301050313, -1.6407106159316551], [0.4903829520027167, -0.20090361729004286, -0.576869080273334, -1.4643903893953993], [-0.3093192834614852, 0.06983846421884547, 0.24498983312506564, -0.8101451679827663], [-0.011265536390620338, -0.062237970028345205, -0.32361960952245516, -1.0995639591685034], [-0.3956122655952703, -0.35460986932582395, 0.04268145688016278, -0.2009729495343217], [-0.835247277155348, -0.16239903472462947, -0.03783854010227068, -0.5641539763808261], [-0.45330687362673494, -0.05902101104513012, 0.3257578800035681, -0.5091902832778865], [-0.49190820450757694, -0.3913114048723003, -0.10825951596097314, -0.0736881438091862], [-1.2182915552819797, -0.025620870876645566, -0.4481222418544762, -0.5423775762333075]]]], "dtype": "float64"}, {"__nd__": [[[1.2939670984901446, 0.31160996111578254, 0.4474041462235054, 1.9000986670869409, 2.0762370293027086, 1.628696098545963, 2.832893684279044, 1.9774772495343493, 4.4576598619699235]], [[0.0634916296988996, 1.540237243002292, 2.120783449756531, 1.3788358248465005, 2.962120227592512, 1.8773738888419018, 3.6150028687634177, 2.3654427073891044, 2.651682425547868]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-2.185237588783683, -2.611978726098935, 12.37576169091523, -20.497864726920493], [3.1414357090251133, -7.958384882548713, -5.531835762921122, -26.12271671847868], [-13.358164587048117, -0.3074089418567264, 13.32679761140413, 14.850398841120757], [-1.2048971249386717, -2.839325358434546, -12.460187805245095, 6.0353474546420856], [-3.7136700076130746, 9.08826808723506, 26.275302061889427, -0.48562435892720496], [-22.46031711261108, -12.912756645432712, -21.87221162004926, -17.956645341224675]], [[19.57906085228202, 3.399893601877243, -22.802616473636387, 9.804482490451827], [18.905623111183235, -5.39811836153793, -10.13986258705329, -5.060665687274238], [4.269097653812572, 9.599552253845566, 17.733252570603504, 18.187173088697488], [17.78105772340468, 20.428821476701096, 9.920816847005302, -22.96359906497979], [-1.9644634427042742, 4.708807689350765, -5.746893975115498, 13.668912823529547], [-5.403209582782572, -13.886205449455652, 21.664484991585933, 9.990064647095066]]]], "dtype": "float64"}, {"__nd__": [[[[3.2888067630057147, 13.900636371897598, 15.938013590030785, 5.115537164590435], [-36.864921283414404, -10.133841760015821, -6.848686943674636, -14.763013426697023], [2.9801615088047284, 17.883031992770125, -7.273583778106025, -17.0224900450631], [30.420522163657495, -6.771564091462998, -18.494473870698766, 3.5651587442768826], [6.385238723843012, -10.575751345210103, 11.856803618823994, -7.309791574300112], [-13.84426373085583, 2.678863027872887, 11.561534656874256, -9.507772367920492]], [[4.959383340213266, -1.402374591080248, 42.36650474604215, -10.628395217446371], [20.762574008092685, -0.7851943265832905, -1.9014300550843524, 10.824976673046567], [13.409858498107535, 19.005234614300875, 4.929776027027368, -9.00594584636888], [-8.06469156276094, 7.644797863250247, 8.713498059116118, 20.97076411044742], [6.276456831280443, 15.902523978866077, 22.751565521759357, 2.472008167966588], [-22.283379773860283, -17.6891562975087, -21.567043022857682, 23.87385571521149]]]], "dtype": "float64"}, {"__nd__": [[[[-0.8474085948289408, 1.2318415654122892, 0.5857319115638746, 1.7152691268390476], [1.0030938194878394, -0.10237026121340073, -0.20003045039326856, 0.0851677199461466], [0.17526095189885732, -0.5307436711234165, -0.03262829758865476, 1.6093200620023946], [-1.7021383108256534, 0.258488139526861, -0.9069528059807551, 0.18703708280488074], [0.8398557752082007, -0.05761018045769089, 0.7729869198484017, -1.5898394013841861], [1.1069891126654727, -0.6031292426481158, 0.6224080697923652, 0.18244809597646958]], [[-2.585562179123786, -0.7560641214702215, 0.22050836793540218, 1.5539760815691257], [0.2872682859036707, 0.25769587977073977, -1.412684089629386, 1.4311466965234902], [1.8070769951881158, 0.02857504312397509, -0.2197570482201158, -1.6190497158096595], [0.8833210589564918, 2.7022793256924214, 0.7171130747864916, 1.2637035181531975], [0.542087293708166, -0.9850158865687554, 1.3366726310198633, -1.2345748994697192], [-0.21066490566304208, 0.2788517450025256, 0.8829096607766574, 0.41598566543096366]]]], "dtype": "float64"}, 2, 2, true], "e": {"__tup__": [{"__nd__": [[[[-0.8474085948289408, 1.2318415654122892, 0.5857319115638746, 1.7152691268390476], [1.0030938194878394, -0.10237026121340073, -0.20003045039326856, 0.0851677199461466], [-0.8474085948269438, 1.2318415654108492, 0.5857319115630267, 1.7152691268372886], [-1.7021383108256534, 0.258488139526861, -0.9069528059807551, 0.18703708280488074], [-0.8474085948289408, 1.2318415654122892, 0.5857319115638746, 1.7152691268390476], [1.0030938194878394, -0.10237026121340073, -0.20003045039326856, 0.0851677199461466]], [[-2.585562179123786, -0.7560641214702215, 0.22050836793540218, 1.5539760815691257], [0.2872682859036707, 0.25769587977073977, -1.412684089629386, 1.4311466965234902], [-2.585562179123786, -0.7560641214702215, 0.22050836793540218, 1.5539760815691257], [1.8070769951881158, 0.02857504312397509, -0.2197570482201158, -1.6190497158096595], [0.8833210589564918, 2.7022793256924214, 0.7171130747864916, 1.2637035181531975], [-2.585562179123786, -0.7560641214702215, 0.22050836793540218, 1.5539760815691257]]]], "dtype": "float64"}, {"__nd__": [[[24.446239284040228, 194.18803097785337, 120.08242333472376, 117.26733174390144, 265.2055293914008, 686.8718906548986], [-488.9704650276848, 180.63331119468592, 282.85269466798866, 441.2061738768713, 144.20633584206374, 402.17361249986163]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.3868345008334123, -0.7614880684287629], [-1.289371413960456, 0.061480668263211365], [-0.7101947772428983, -1.3077856280692512], [-1.265946471501941, -0.49012847417311295], [-1.852578860174007, -1.3474687028443724]]]], "dtype": "float64"}, {"__nd__": [[[[-1.6351369290692526, 0.1822594637816633], [0.4079842295184183, 2.0071409631828265], [-1.4975297992323828, -0.6795969589096627], [0.9126113893228037, -0.2167738049889562], [-0.3269979387077995, 1.709169703339553]]]], "dtype": "float64"}, {"__nd__": [[[[-0.3384061595207278, -1.1561948693340067, -1.3169399598669396], [0.3356943223139325, 0.30524160340596707, -1.3717023702861768], [-0.9853962577263109, 0.5347126378156121, 0.5772667725251409], [-0.6432220836039892, 0.6222162018519841, 0.5718545872905609], [-1.7830817487195436, -0.3120637504487101, 0.41512559112112213]]]], "dtype": "float64"}, 8, 8], "e": {"__tup__": [{"__nd__": [[[[-0.6955257470378747, 0.19912759353851014, 0.06579641627253369], [-0.7215484577341077, -0.28374032528969617, -0.35571240376248897], [-0.7749342285050483, 0.049706088438985774, 0.01909457535760829], [-0.7315679092780502, -0.1850113040748785, -0.23397476366188347], [-0.7699790437936418, -0.03937509091293591, -0.06343072096815824]]]], "dtype": "float64"}, {"__nd__": [[[1.3145383789355518, 2.3900006494334756, 1.9516101199109328, 2.347248737219865, 3.0683841341577587]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-0.5730757021639871, -2.138931868967707, -0.2830147214661213, 0.7520931599414118, 1.5804055858131327, -2.190782661253991], [2.6133213455642696, -1.1271363310292688, 0.9569708708011316, 1.2341980705343905, 0.9830173001338016, 0.14961847956037583], [-1.1614176626879231, 0.6265587879860715, -0.7152331767755107, -2.5226949434372874, 2.8288986337162725, 0.7081020219606693], [1.8180771284865394, -0.9116228074944426, 1.4675028234118994, 1.5898691581489963, 1.5545495245011742, -0.0212063332121524], [-1.2024620437425992, -0.17731280686363585, -2.2027691137666046, -1.702012389403535, 0.09193878115026889, -0.991398465768181], [-0.16444685545475027, -0.10899975140135905, 1.9307270485184007, 1.297626569750945, -0.04487258416282581, 1.2088995341693431], [-0.7797131446903424, -0.33689702409487016, 0.9102243101173967, -1.2416605775709948, -0.2789807283002251, -1.3208458600880442], [0.11747240782797058, 1.6325157555894336, -0.7520968891009723, 0.22587949591382886, -0.8973219016364986, -1.1495489432403887], [-1.235849520341746, 1.4547520814096038, 2.362435470617594, 0.4830437053055756, -0.7162929691373378, 0.7048916343431867], [-0.3343235263430333, 0.8013983475373727, 0.29099363819943935, 0.27970415085508915, 0.6072760593752837, -0.5794936369771572]], [[-0.4043678231128611, -1.7212455900539538, -0.441869441453459, -1.422332690064836, -0.14468119903994156, -0.9576309493862568], [0.10123442032130817, 0.45323930641573995, -1.420122571064877, -0.8132790942459649, -0.9433119018353825, 0.7464934250784422], [1.7063620273868247, 0.8515366153697317, -0.34941892769912597, 1.4650063896361285, -1.5141077364412707, 1.4993580521280918], [-0.6562142063018171, -2.7802932151547175, 2.6508047390847573, 1.5780230743235615, -1.0318837271201073, 0.16900864759315984], [-0.21449945225320038, 0.10222080329493188, -1.4832864375412333, -0.681493982666794, 0.46796128274761895, 0.2226961699555749], [1.1992385100066276, 0.07975025742709378, 2.3492216858979997, -0.7157315571686473, 0.2593323919535436, -1.916317895933467], [-1.2531331287645449, 1.3285259609313695, -0.9428027674786568, 0.5306751740014318, -0.06177085735139875, -1.0445489576875715], [-0.34824927418715956, -0.5142336346467203, -0.4873658684579376, 0.753608210927443, 0.6523445544165802, -0.5774256966270559], [-0.907102803839221, 0.2823373345183147, -0.10001541175059649, 1.0282824624497804, 2.655293570613392, 0.8429244344247891], [-0.034730271064206376, -0.17204397087905782, -0.8566941204641656, -0.9066727843825847, -1.003415510501075, -0.39533517140537894]]]], "dtype": "float64"}, {"__nd__": [[[[-0.42714344576083985, 0.7437307065618273, -0.39899901500045487, -0.1974497164360777, -1.2182740255558127, 1.644575629197004], [0.5069864095865559, 1.7888731692239133, 0.7968857920360226, 1.489635951913291, -0.24848752544496672, 0.7171608079569028], [2.705506369509455, -1.2235344616058148, 1.1762689550642216, 1.6979442184119353, 0.4211275777094194, 0.7652124357168802], [1.279634467713052, -0.6702951659654681, 0.4174117281438917, -0.888004935618753, -0.6912470413214244, -0.6315627456115833], [-0.14626827269219284, 0.14977563848346317, 0.4630331670186004, -1.4509086898153556, 2.0267754218405436, 1.183528956756262], [0.7435627693927573, -0.36653039160670736, -0.09130666673423081, 0.5611664206332558, 0.42143998639528435, -1.7210624380768245], [0.7592792642700629, 2.9882475884033077, -1.8932899501520968, 1.019677621634092, 0.37850197507595706, -0.11441521971153995], [1.4070157400223116, -1.213596313080212, 0.16709499277851292, 1.5019546519282645, -0.2020985448325452, -0.42926303302657054], [0.11797722239794746, -0.45675534800482764, 1.5891687302935997, -0.931847773671157, 0.8969312703305735, -1.286683799566433], [0.6875724027218497, -0.10779946957536914, -2.65516197132188, -0.006143153353732409, -1.0967837412742523, -0.45678151259372246]], [[1.581551798649239, -1.1485810870406674, -1.0899504257265902, 1.4808828568718004, 0.11919692046819659, 1.5847309175915665], [0.33260737411589053, -0.90706467160641, -0.06816737185530974, 1.2641284304472291, 0.9675052248511342, 0.0009115320758069846], [0.05014367071243192, -0.11570044011517851, -0.5946671712463415, 1.9858797493751439, 0.007393170164879609, -1.1629091187923692], [-0.49041754861728876, 0.7468198607642406, 0.6595594318782978, -0.14099903467320576, -0.6872569566347936, 0.024582120939776186], [-1.7206875500530123, -1.3384432150414223, 0.821399594139533, -0.4927687948123542, 0.385483933892715, 1.252592533070883], [0.6679124924998421, 0.021189653318113826, 2.187981864530712, 0.6864987964126616, -0.36446034394198246, 0.7526967264568842], [0.419346493896769, -0.8799859722813214, 0.07501520407748195, -0.16962420039136794, -2.0215523437034415, -0.17725145727058092], [-0.29105534618990875, -0.45063933502457765, -1.691096798191634, -0.3646500182510735, -0.04080209072452791, 0.10479554070860274], [-1.1673596004573077, 0.6502266590909276, -1.2496713584632424, -0.23004121509448516, 1.341658720456726, -0.7117573020089456], [-0.5250016779687878, 1.1033530773375757, -0.3947134948174819, -0.3052563851455138, 0.22735868500971498, 1.0784191064946678]]]], "dtype": "float64"}, {"__nd__": [[[[-2.2646259109564686, -0.8511711525341055, -1.0349134408139684, -1.1144134810834543, -0.8777834753447102, -0.8615483089052315], [0.3092160512002342, -0.9121276954716766, 0.12558427796144458, 0.41142057487189804, -0.7480531806818757, 0.1338761754133952], [1.6775191154460816, 0.33005396421210825, -0.6529794238599105, -0.06652061107914677, -0.9379420327921617, 0.27713621736140426], [0.8824491394473416, -0.8379442947522655, -0.21394383528031954, 1.1372961410537146, -0.5084857091926813, 0.19210250301024473], [-1.046130800054778, -0.8367857772429679, -0.52346318956414, 2.176320023214294, -0.40905014133652295, -0.5299586860696227], [-0.563193528477139, -0.16560235405827292, 0.6559301183529251, 0.18949043700352977, 2.0578146747003956, 0.21379444673291043], [1.4474041493565695, 0.2767111160248597, 0.6055814984474479, -1.4435999397133137, -0.12705944813917863, -0.06894774062879866], [-0.24295700689747154, -2.152878627190867, 0.8806244618726567, -0.3871503296821815, -0.4380713222463156, -0.25517146104015803], [-0.22807918887443684, 0.40709842078875075, -1.2710322515652643, -0.9310420569169968, 0.2906332958993996, 0.1455828113203634], [-0.37665480890873254, -0.5739124056409823, -0.8344247031699268, -0.0583551638956689, 1.1807324731914441, -1.0580507513790518]], [[1.5233180466501626, 0.8439814653804283, -0.45296373730993883, 0.7830748754630861, -1.4659034168620872, -1.7315749451612408], [-1.2897822561511796, -0.4097464653141918, -0.3254099689496813, -0.51514241714267, 0.09942269056475599, -2.1351539077186805], [0.4953494063515442, -1.4193005788923314, -0.6608506052253199, -0.4128504662929992, -1.156210636742483, -1.3709996686375883], [-1.0060816652143074, -0.0119236421411786, 0.5899821790594377, 0.5642351777039467, -0.9944910382704913, -0.791577148211955], [-0.9206031646499282, 0.44981231209647754, -2.037420180518552, 1.2383392605822288, -0.48480487528966326, -0.7835248638003831], [0.3394537959075636, 1.0326190551063525, 0.34258325688492597, 1.0527444573186129, 0.13435881665376626, 1.2296386057077333], [1.245160038618696, -0.09947878933206503, 1.4392058879269531, 0.16473370131349793, 0.08689322030202752, -0.9214736704473936], [-0.28878000578719426, 0.7775827458435827, -1.3126469097364055, 0.6496240093252744, 0.34861595126946426, 0.2775614162501203], [-2.2064921954239676, -0.1032429196345987, 0.648260007690957, -0.7629549569181814, 0.090306699097783, -2.3780732690682194], [0.6713930362884684, -0.6624328096757435, 0.8713569475667259, -1.8427862759322255, 0.692267388325644, -0.2369392876523166]]]], "dtype": "float64"}, 1, 1, true], "e": {"__tup__": [{"__nd__": [[[[-2.2646259109564686, -0.8511711525341055, -1.0349134408139684, -1.1144134810834543, -0.8777834753447102, -0.8615483089052315], [0.044386910953446115, -0.9058557218633798, 0.00617771856875871, 0.25442363133798745, -0.7614014603760697, 0.03145442060533735], [-1.5935221249991474, -0.8054427415747145, -0.7997934300185829, -0.7788914046946683, -0.8564724440318394, -0.6206595495689555], [1.6198793340695035, 0.2748959048496899, -0.625093051288092, -0.03594601860694485, -0.9257211698957623, 0.2697737376655043], [-0.8097541133552265, -0.835042060508255, -0.5698368097776281, 0.865384685550148, -0.5861836265125356, -0.40241189785611015], [0.6206848390431551, -0.2968659645006103, -0.38100979771341, 0.19264387642420117, -0.7314925219811731, 0.0715615407407905], [-0.09407597472933393, -0.5970878211264021, -0.0923560978723617, 0.7520278349434175, 0.12018945059307425, -0.031638910897778164], [0.8293007096728043, -0.10056764962162502, 0.40117635296854887, -0.8800792415640705, -0.10769584165882157, -0.06710351227129059], [-0.1976249606567428, -0.6944471697893445, -0.2382938841198844, 0.04089417754444992, -0.5710345277004966, -0.09296582165534861], [0.1220466224604857, -0.3682361815482913, -0.08797221068966911, -0.14896473285897968, 0.013748258939334634, -0.09234399027051811]], [[1.5233180466501626, 0.8439814653804283, -0.45296373730993883, 0.7830748754630861, -1.4659034168620872, -1.7315749451612408], [0.9380516484391759, 0.5831429771105803, -0.42642613589128137, 0.5129803615269984, -1.1402368304246862, -1.8155396746003245], [1.235546420557786, 0.5297295048038465, -0.46613848092402793, 0.5759259029371647, -1.3328035505993405, -1.7205108759276675], [-0.19994658682841265, -0.28793218600255105, -0.3027087512980666, -0.04116152933046192, -0.6951638432166448, -1.6851129096986506], [-0.06830492463047914, 0.019431116185719955, -0.6403014804983562, 0.40267478391959294, -0.8440660533619463, -1.406762843972811], [-0.06954232947093995, 0.4574435843698038, 0.09932707211521294, 0.6872667877480044, -0.23609490556094848, 0.1782344723733804], [-0.026867250401573485, -0.5791759931223933, -0.22069772474904917, 0.05928133580037847, -0.8029594716081411, -1.153882752285941], [-0.03145508687266916, -0.15353690226565192, -0.5255056329088948, 0.18893843710769173, -0.5235389506296245, -1.1128344828844428], [-0.9070400373998877, 0.010064113471904019, -0.348316682194234, 0.016464140734112478, -0.28906378285906825, -1.5438554642378537], [0.005503765264674656, 0.06859801992728383, 0.07298134361951825, 0.10895052404526882, -0.13046642941018083, -0.8029856268755756]]]], "dtype": "float64"}, {"__nd__": [[[-2.8209371503279357, 0.8323016796805652, 0.0489934542823913, 4.577390383217151, 1.173847006590988, 2.8319446392022374, 1.736648758203175, 3.0529289848321643, 2.7969228919508833, 2.4365481521877754], [-0.7438544172752447, 0.6633750988778316, 2.8235184092600547, 2.1734587650900257, 1.4823390812420636, 2.1466871816979207, 1.8774925180792756, 2.366618634865505, 2.8919007474543497, 2.367170883991891]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(flash_attention, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")